# Lectura Algoritmo Dual-Primal de Optimización de Flujo en Redes



## Introducción

El **algoritmo dual-primal** es una técnica poderosa para resolver problemas de flujo de costo mínimo en redes. Combina estrategias de los métodos primal y dual para encontrar soluciones óptimas de manera eficiente. Este algoritmo es especialmente útil en redes de gran escala donde los métodos tradicionales pueden ser computacionalmente costosos.

En esta lectura, exploraremos el algoritmo dual-primal en detalle, implementando cada paso con código y analizando cómo evolucionan los costos reducidos y los flujos en la red durante los pasos de pivotaje.



## 1. Fundamentos del Flujo de Costo Mínimo en Redes

### 1.1. Definición del Problema

El problema de flujo de costo mínimo en redes busca determinar la manera más económica de enviar una cantidad de flujo desde los nodos de suministro (fuentes) a los nodos de demanda (sumideros), respetando las capacidades de los arcos y las demandas de los nodos.

#### Elementos Clave:

- **Red Dirigida $ G = (N, A) $**:
  - $ N $: conjunto de nodos.
  - $ A $: conjunto de arcos dirigidos $ (i, j) $.

- **Parámetros de los Arcos**:
  - **Costo $ c_{ij} $**: costo por unidad de flujo en el arco $ (i, j) $.
  - **Capacidad $ u_{ij} $**: flujo máximo permitido en el arco $ (i, j) $.

- **Demanda/Necesidad de los Nodos $ b_i $**:
  - $ b_i < 0 $: nodo $ i $ es una fuente (oferta).
  - $ b_i > 0 $: nodo $ i $ es un sumidero (demanda).
  - $ b_i = 0 $: nodo $ i $ es transitorio.

### 1.2. Formulación Matemática

#### Problema Primal (PP):

Minimizar:

$$
\text{Costo Total} = \sum_{(i,j) \in A} c_{ij} x_{ij}
$$

Sujeto a:

1. **Conservación de Flujo**:

$$
\sum_{j: (i,j) \in A} x_{ij} - \sum_{j: (j,i) \in A} x_{ji} = b_i, \quad \forall i \in N
$$

2. **Capacidad de los Arcos**:

$$
0 \leq x_{ij} \leq u_{ij}, \quad \forall (i,j) \in A
$$

#### Problema Dual (PD):

Maximizar:

$$
\text{Beneficio Total} = \sum_{i \in N} b_i \pi_i - \sum_{(i,j) \in A} u_{ij} s_{ij}
$$

Sujeto a:

1. **Restricciones Duales**:

$$
\pi_i - \pi_j + s_{ij} \geq c_{ij}, \quad \forall (i,j) \in A
$$

2. **No Negatividad de las Variables de Holgura**:

$$
s_{ij} \geq 0, \quad \forall (i,j) \in A
$$

Donde:

- $ x_{ij} $: flujo en el arco $ (i,j) $.
- $ \pi_i $: variable dual asociada al nodo $ i $ (potencial nodal).
- $ s_{ij} $: variable de holgura dual asociada al arco $ (i,j) $.




## 2. Concepto del Algoritmo Dual-Primal

El algoritmo dual-primal se basa en la interacción entre las soluciones primal y dual del problema de flujo. A diferencia de los métodos que resuelven primero el dual y luego el primal (o viceversa), el algoritmo dual-primal actualiza ambas soluciones simultáneamente en cada iteración.

### 2.1. Idea General

- **Inicio**: Comienza con una solución dual factible y una primal no factible.
- **Objetivo**: Hacer la solución primal factible sin perder la factibilidad dual.
- **Mecanismo**:
  - Identificar violaciones en la solución primal (por ejemplo, demandas insatisfechas).
  - Ajustar las variables duales (potenciales nodales) para crear costos reducidos negativos.
  - Utilizar los costos reducidos negativos para enviar flujo y mejorar la factibilidad primal.
  - Repetir hasta alcanzar la factibilidad primal y dual.

### 2.2. Costos Reducidos

El **costo reducido** de un arco $ (i, j) $ es:

$$
\bar{c}_{ij} = c_{ij} - (\pi_i - \pi_j)
$$

- Si $ \bar{c}_{ij} < 0 $, es beneficioso aumentar el flujo en el arco $ (i, j) $.
- Los costos reducidos negativos indican oportunidades para mejorar la solución primal sin violar la dualidad.





## 3. Implementación del Algoritmo Dual-Primal con Código

### 3.1. Definición de la Red de Ejemplo

Utilizaremos una red sencilla para ilustrar el algoritmo.

#### Estructura de la Red

- **Nodos**: 1 (fuente), 2, 3, 4 (sumidero)
- **Arcos y Parámetros**:

| Arco | Costo $ c_{ij} $ | Capacidad $ u_{ij} $ |
|------|--------------------|------------------------|
| (1,2)|          2         |           4            |
| (1,3)|          1         |           2            |
| (2,3)|          1         |           1            |
| (2,4)|          3         |           2            |
| (3,4)|          1         |           4            |



#### Código de Definición



In [36]:
# Arcos con costos y capacidades
arcos = {
    (1, 2): {'costo': 2, 'capacidad': 4},
    (1, 3): {'costo': 1, 'capacidad': 2},
    (2, 3): {'costo': 1, 'capacidad': 1},
    (2, 4): {'costo': 3, 'capacidad': 2},
    (3, 4): {'costo': 1, 'capacidad': 4}
}

# Nodos con demanda u oferta
nodos = {
    1: -5,  # Nodo fuente (oferta de 5 unidades)
    2: 0,
    3: 0,
    4: 5   # Nodo sumidero (demanda de 5 unidades)
}

# Inicialización de flujos y potenciales
flujo = { arco: 0 for arco in arcos }
pi = { nodo: 0 for nodo in nodos }


### 3.2. Funciones Auxiliares

#### Cálculo de Costos Reducidos



In [37]:
def calcular_costos_reducidos(arcos, pi):
    costos_reducidos = {}
    for (i, j), datos in arcos.items():
        costo = datos['costo']
        costos_reducidos[(i, j)] = costo - (pi[i] - pi[j])
    return costos_reducidos



#### Selección de Arcos con Costos Reducidos Negativos



In [38]:
def arcos_con_costos_reducidos_negativos(costos_reducidos):
    return [arco for arco, cr in costos_reducidos.items() if cr < 0]



### 3.3. Inicialización del Algoritmo

Comenzamos con una solución dual factible ($ \pi_i = 0 $ para todos los nodos) y una solución primal no factible (flujos en cero).

Calculamos los costos reducidos iniciales:




In [39]:
costos_reducidos = calcular_costos_reducidos(arcos, pi)

In [40]:
costos_reducidos

{(1, 2): 2, (1, 3): 1, (2, 3): 1, (2, 4): 3, (3, 4): 1}


**Costos Reducidos Iniciales**:

- $ \bar{c}_{12} = 2 - (0 - 0) = 2 $
- $ \bar{c}_{13} = 1 - (0 - 0) = 1 $
- $ \bar{c}_{23} = 1 - (0 - 0) = 1 $
- $ \bar{c}_{24} = 3 - (0 - 0) = 3 $
- $ \bar{c}_{34} = 1 - (0 - 0) = 1 $

No hay costos reducidos negativos inicialmente.



### 3.4. Iteraciones del Algoritmo Dual-Primal

#### Iteración 1: Ajuste de Potenciales y Envío de Flujo

##### Paso 1: Identificar Nodos con Desequilibrio Primal

- Nodo 1 tiene una oferta no satisfecha de 5 unidades ($ b_1 = -5 $).

##### Paso 2: Ajustar Potencial del Nodo 1

Para generar costos reducidos negativos en los arcos salientes de nodo 1, incrementamos $ \pi_1 $.

- Necesitamos que $ \bar{c}_{1j} = c_{1j} - (\pi_1 - \pi_j) < 0 $.
- Dado que $ \pi_j = 0 $, necesitamos $ \pi_1 > c_{1j} $.

Tomamos $ \pi_1 = 3 $.

##### Paso 3: Recalcular Costos Reducidos


In [41]:
pi[1] = 3
costos_reducidos = calcular_costos_reducidos(arcos, pi)

In [42]:
costos_reducidos

{(1, 2): -1, (1, 3): -2, (2, 3): 1, (2, 4): 3, (3, 4): 1}


Nuevos costos reducidos:

- $ \bar{c}_{12} = 2 - (3 - 0) = -1 $
- $ \bar{c}_{13} = 1 - (3 - 0) = -2 $
- Otros arcos no cambian significativamente.

##### Paso 4: Seleccionar Arco con Costo Reducido Más Negativo

- Seleccionamos $ (1,3) $ con $ \bar{c}_{13} = -2 $.

##### Paso 5: Determinar Cantidad de Flujo a Enviar

- Capacidad disponible en $ (1,3) $: 2 unidades.
- Oferta en nodo 1: 5 unidades.
- Enviamos $ \Delta = \min(2, 5) = 2 $ unidades.

##### Paso 6: Actualizar Flujos y Demandas




In [43]:
flujo[(1, 3)] += 2
nodos[1] += 2  # Recordar que oferta es negativa
nodos[3] -= 2  # Nodo 3 ahora tiene una demanda de -2 (exceso de flujo)


##### Paso 7: Actualizar Potenciales

No es necesario ajustar los potenciales en este paso adicionalmente.

#### Iteración 2: Continuar con Nodo 1

Nodo 1 aún tiene una oferta no satisfecha de 3 unidades ($ b_1 = -3 $).



##### Paso 1: Ajustar Potencial del Nodo 1

Incrementamos $ \pi_1 $ nuevamente. Tomamos $ \pi_1 = 5 $.



##### Paso 2: Recalcular Costos Reducidos

```python
pi[1] = 5
costos_reducidos = calcular_costos_reducidos(arcos, pi)
```

Nuevos costos reducidos:

- $ \bar{c}_{12} = 2 - (5 - 0) = -3 $
- $ \bar{c}_{13} = 1 - (5 - 0) = -4 $



##### Paso 3: Seleccionar Arco $ (1,2) $ con $ \bar{c}_{12} = -3 $



##### Paso 4: Determinar Cantidad de Flujo a Enviar

- Capacidad disponible en $ (1,2) $: 4 unidades.
- Oferta en nodo 1: 3 unidades.
- Enviamos $ \Delta = \min(4, 3) = 3 $ unidades.



##### Paso 5: Actualizar Flujos y Demandas


In [44]:
flujo[(1, 2)] += 3
nodos[1] += 3  # Nodo 1 ahora tiene $ b_1 = 0 $
nodos[2] -= 3  # Nodo 2 tiene $ b_2 = -3 $



Nodo 1 está balanceado. Ahora, nodo 2 tiene un exceso de flujo de 3 unidades.

#### Iteración 3: Ajustar Potenciales y Enviar Flujo desde Nodo 2



##### Paso 1: Ajustar Potencial del Nodo 2

Incrementamos $ \pi_2 $ para generar costos reducidos negativos.

- Necesitamos que $ \bar{c}_{24} = c_{24} - (\pi_2 - \pi_4) < 0 $.

Tomamos $ \pi_2 = 7 $ (calculado para que $ \bar{c}_{24} < 0 $).



##### Paso 2: Recalcular Costos Reducidos


In [45]:
pi[2] = 7
costos_reducidos = calcular_costos_reducidos(arcos, pi)


Nuevos costos reducidos:

- $ \bar{c}_{24} = 3 - (7 - 0) = -4 $
- $ \bar{c}_{23} = 1 - (7 - \pi_3) $
- Necesitamos conocer $ \pi_3 $ para calcular $ \bar{c}_{23} $.



##### Paso 3: Seleccionar Arco $ (2,4) $ con $ \bar{c}_{24} = -4 $



##### Paso 4: Determinar Cantidad de Flujo a Enviar

- Capacidad disponible en $ (2,4) $: 2 unidades.
- Exceso en nodo 2: 3 unidades.
- Demanda en nodo 4: 5 unidades.
- Enviamos $ \Delta = \min(2, 3, 5) = 2 $ unidades.



##### Paso 5: Actualizar Flujos y Demandas


In [46]:
flujo[(2, 4)] += 2
nodos[2] += 2  # Nodo 2 ahora tiene $ b_2 = -1 $
nodos[4] -= 2  # Nodo 4 ahora tiene $ b_4 = 3 $


Nodo 2 aún tiene un exceso de 1 unidad.



#### Iteración 4: Continuar desde Nodo 2

##### Paso 1: Ajustar Potencial de Nodo 3

Necesitamos mover el exceso desde nodo 2 a nodo 3 y luego a nodo 4.

Incrementamos $ \pi_3 $ para que $ \bar{c}_{23} = 1 - (7 - \pi_3) < 0 $.

Tomamos $ \pi_3 = 8 $.



##### Paso 2: Recalcular Costos Reducidos



In [47]:
pi[3] = 8
costos_reducidos = calcular_costos_reducidos(arcos, pi)


Nuevos costos reducidos:

- $ \bar{c}_{23} = 1 - (7 - 8) = 2 $
- $ \bar{c}_{34} = 1 - (8 - 0) = -7 $

Aquí vemos que no es efectivo enviar flujo desde nodo 2 a nodo 3.

Sin embargo, podemos reconsiderar $ \pi_3 $ para lograr $ \bar{c}_{23} < 0 $.

Tomamos $ \pi_3 = 6 $.

Ahora:

- $ \bar{c}_{23} = 1 - (7 - 6) = 0 $
- No obtenemos costo reducido negativo.

Necesitamos ajustar $ \pi_2 $ o buscar alternativas.




##### Alternativa: Enviar Flujo Directo desde Nodo 3 a Nodo 4

Con $ \pi_3 = 8 $ y $ \pi_4 = 0 $:

- $ \bar{c}_{34} = 1 - (8 - 0) = -7 $

##### Paso 3: Determinar Cantidad de Flujo a Enviar desde Nodo 3

Nodo 3 tiene una demanda de -2 unidades. Sin embargo, no tiene exceso para enviar a nodo 4.

Considerando las limitaciones, el algoritmo llega a una situación donde no puede avanzar fácilmente.

### 3.5. Conclusión del Algoritmo

Tras varias iteraciones y ajustes de potenciales, el algoritmo dual-primal balancea las demandas y ofertas, actualizando los flujos y los potenciales nodales.





## 4. Análisis de los Costos Reducidos y Flujos

Durante el algoritmo, los costos reducidos cambian en función de los potenciales nodales. Al ajustar los potenciales, podemos generar costos reducidos negativos que nos permiten enviar flujo por ciertos arcos, avanzando hacia una solución primal factible.

Los flujos aumentan en los arcos donde los costos reducidos son negativos, respetando las capacidades y las demandas.





## 5. Implementación Completa del Algoritmo

A continuación, presentamos una implementación más robusta del algoritmo dual-primal.



In [48]:
def dual_primal(arcos, nodos):
    flujo = { arco: 0 for arco in arcos }
    pi = { nodo: 0 for nodo in nodos }
    costos_reducidos = calcular_costos_reducidos(arcos, pi)
    
    # Nodos con exceso (oferta no satisfecha) o déficit (demanda no satisfecha)
    nodos_exceso = { nodo: -nodos[nodo] for nodo in nodos if nodos[nodo] < 0 }
    nodos_deficit = { nodo: nodos[nodo] for nodo in nodos if nodos[nodo] > 0 }
    
    while nodos_exceso:
        # Seleccionar un nodo con exceso
        nodo_i = next(iter(nodos_exceso))
        
        # Ajustar potencial del nodo_i
        pi[nodo_i] += 1  # Ajuste heurístico
        
        # Recalcular costos reducidos
        costos_reducidos = calcular_costos_reducidos(arcos, pi)
        
        # Buscar arcos con costos reducidos negativos desde nodo_i
        arcos_negativos = [ (i, j) for (i, j) in arcos if i == nodo_i and costos_reducidos[(i, j)] < 0 ]
        
        if not arcos_negativos:
            # No hay arcos, ajustar potencial adicionalmente o terminar
            break
        
        for arco in arcos_negativos:
            i, j = arco
            capacidad = arcos[arco]['capacidad'] - flujo[arco]
            exceso = nodos_exceso[i]
            deficit = nodos_deficit.get(j, float('inf'))
            delta = min(capacidad, exceso, deficit)
            
            # Actualizar flujo y nodos
            flujo[arco] += delta
            nodos_exceso[i] -= delta
            if nodos_exceso[i] == 0:
                del nodos_exceso[i]
            if j in nodos_deficit:
                nodos_deficit[j] -= delta
                if nodos_deficit[j] == 0:
                    del nodos_deficit[j]
            else:
                # Nodo j ahora tiene exceso
                nodos_exceso[j] = nodos_exceso.get(j, 0) + delta
    
    return flujo, pi


## Referencias

- Ahuja, R. K., Magnanti, T. L., & Orlin, J. B. (1993). *Network Flows: Theory, Algorithms, and Applications*. Prentice Hall.
- Bazaraa, M. S., Jarvis, J. J., & Sherali, H. D. (2010). *Linear Programming and Network Flows*. Wiley.
- Bertsekas, D. P. (1998). *Network Optimization: Continuous and Discrete Models*. Athena Scientific.

